In [ ]:
import numpy as np 
from pathlib import Path

def load_gpt2_weights(cach_dir: Path):
    ckpt=tf.train.load_checkpoint(str(cach_dir/ "gpt2_124m"/ "model.ckpt"))
    weights= { name : ckpt.get_tensor(name) 
               for name in ckpt.get_variable_to_shape_map()
    }
    return weights

In [ ]:
import json , regex as re

class BytePairTokenizer:
    def __init__(self, encoder_path:Path, bpe_path:Path):
        self.encoder= json.loads(encoder_path.read_text())
        self.decoder= {v:k 
                       for k,v in self.encoder.items()}
        with bpe_path.open() as f : 
            bpe_merges= [tuple(line.rstrip().split())
                        for line in f if line and not line.staartwith("#")]
            self.bpe_ranks= { pair : i 
                             for i , pair in enumerate(bpe_merges)}
            self.pat= re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+""")

    def bpe(self,token):
        word= tuple(token)
        pairs = {tuple(word[i:i+2])
                 for i in range (len(word)-1)}
        while True:
            # from all the bigrams currently available, find the one with the lowest bpe_rank
            bigram= min(pairs,
                        key=lambda pair : self.bpe_ranks.get(pair, float("inf")), 
                        default=None
                        )
            if bigram is None or bigram not in self.bpe_ranks:
                break 
            first , second = bigram
            new_word = [first]
            i=1 
            while i < len(word):
                if i < len(word)-1 and word[i]== first and word[i+1]== second :
                    new_word.append(first + second)
                    i+=2
                else : 
                    new_word.append(word[i])
                    i+=1
            word = tuple(new_word)
            pairs = { tuple(word[i:i+2])
                     for i in range (len(word)-1)
            }
        return " ".join(word)
    
    def encode(self,text:str)->list[int]:
        ids=[]
        for tok in re.findall(self.pat, text):
            tok = " ".join(self.byte_encoder[b]
                           for b in tok.encode("utf-8")
                        )
            ids.extend(self.encoder[pair]
                       for pair in self.bpe(tok).split()
                       )
        return ids
    
    def decode (self, ids: list[int])->str:
        text= " ".join(self.decoder[id] for id in ids )
        return bytearray(self.byte_decoder[c] for c in text.decode("utf-8","replace"))


In [ ]:
def gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0/ np.pi))* (x +0.044715 * x**3))

def softmax( x , axis=1):
    x = x - x.max(axis=axis , keepdims=True)
    return np.exp(x) / np.exp(x).sum(axis=axis , keepdims=True)

def layer_norm(x , gamma , beta , eps= le-5):
    mean = x.mean(axis=-1 , keepdims=True)
    var= = x.var(axis=-1 , keepdims=True)
    return gamma * (x-mean) / np.sqrt(var+eps) + beta
def attention(q, k , v , mask ):
    #q,k , v : [T,d]
    
    scores= q @ k.T / np.sqrt(q.shape[-1])   # [T,d] @ [d,T] -> [T,T]
    """
    A → A
    B → A B
    C → A B C
    D → A B C D
    """
    scores= scores + mask   # causal mask
    weights=softmax(scores,axis=-1)  #Convert scores into probabilities , Apply softmax across the last dimension.
    return weights @ v 

def block(x , params , kv_cache=None , position_offset=0):
    #x: [T, d_model] 
    a= layer_norm(x ,params["ln1_gamma"],params["ln1_beta"])
    q= a @ params["wq"]
    k= a @ params["wk"]
    v= a @ params["wv"]

    if kv_cache is not None : 
        k = np.concatenate([kv_cache["k"],k]axis=0)
        v = np.concatenate([kv_cache["v"],v]axis=0)
        kv_cache["k"]=k
        kv_cache["v"]=v

        #split into heads , attend and then recombine , MLP/FFN processes the result 
    x = x + attention(q,k,v , mask=casual_mask(len(k)))
    m = layer_norm( x , params["ln2_gamma"],params["ln2_beta"]) 
    h = gelu( m @ params["ffw1"] @ params["ffw2"])
    return x + h  


In [ ]:
def sample(logits, , temperature=1.0, top_k= 0, top_p= 1.0):
    logits= logits / max ( temperature , 1e-5)
    if top_k : 
        idx = np.argpartition(-logits , top_k )[:top_k]
        mask = np.full_like(logits , -np.inf) 
        mask[idx]= logits[idx]
        logits= mask 
    if top_p > 1.0 :
        sort_idx = np.argsort(-logits)
        sorted = logits[sort_idx]
        probs= softmax(sorted)
        cum = np.cumsum(probs)
        cutoff = cum > top_p 
        cutoff[1:] = cutoff[:-1].copy()
        cutoff[0] = False
        logits[sort_idx[cutoff]] = -np.inf
    probs = softmax (logits)
    return np.random.choice ( len(probs),p=probs)

In [ ]:
def generate(model , tokenizer , prompt , max_new_tokens=64 , **sampling_kwargs):
    tokens = tokenizer.encode(prompt)
    kv_caches = [ {} for _ in model["blocks"]]
    for _ in range(max_new_tokens):
        # prefill 
       x = model["wte"][tokens] + model["wpe"][np.arrange(len(tokens))] #x = token_embeddings + position_embeddings
       for i , block_params in enumerate(model["blocks"]): #Loop through every Transformer blocks in the model
           x = block ( x , block_params , kv_cache=kv_caches[i])
        logits = layer_norm(x , model["ln_f_gamma"],model["ln_f_beta"]) @ model["wte"].T)[-1] 
        nxt= sample( logits , **sampling_kwargs)
        generated.append(nxt)
        if nxt == tokenizer.eos_token_id:  
            break
        tokens = [nxt]
        return tokenizer.decode(generated)   


"""
Prompt
"The cat is"
     ↓
Tokenizer
     ↓
[464, 3797, 318]
     ↓
Token embeddings + Position embeddings
     ↓
x [T, 768]
     ↓
┌─────────────────┐
│ Transformer 0   │
│ Transformer 1   │
│ Transformer 2   │
│ ...             │
│ Transformer 11  │
└─────────────────┘
     ↓
Final LayerNorm
     ↓
x [T, 768]
     ↓
@ wte.T
     ↓
Logits [T, 50257]
     ↓
[-1]
     ↓
Logits [50257]
     ↓
Temperature
     ↓
Top-K
     ↓
Top-P
     ↓
Softmax
     ↓
Probabilities [50257]
     ↓
np.random.choice(...)
     ↓
NEXT TOKEN ID
"""